# LeetCode 1508 — Range Sum of Sorted Subarray Sums bằng Python

Link đề: https://leetcode.com/problems/range-sum-of-sorted-subarray-sums/ · Độ khó: **Medium** · Chủ đề: Array, Two Pointers, Binary Search, Sorting, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def rangeSum(self, nums: list[int], n: int, left: int, right: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Sinh **mọi tổng đoạn con** trong `O(n²)` bằng tổng chạy (running sum) thay vì `sum(nums[i:j])` `O(n³)`.
2. Dùng **heap k-way merge** để lấy lần lượt các tổng nhỏ nhất mà không sinh hết.
3. Tận dụng ràng buộc giá trị nhỏ (`nums[i] ≤ 100`) để **sắp xếp đếm** (counting sort).
4. Nâng cao: **binary search trên giá trị** + **hai con trỏ** để tính "tổng của k tổng nhỏ nhất" trong `O(n log S)`, xử lý đúng trường hợp giá trị trùng.
5. Cẩn thận chỉ số **1-indexed** và phép lấy **modulo 10⁹ + 7**.

## 2. Đề bài & ví dụ

Cho mảng `nums` gồm `n` số nguyên **dương**. Tính tổng của **mọi đoạn con liên tiếp khác rỗng** (có `n(n+1)/2` đoạn), sắp xếp các tổng này tăng dần thành mảng mới. Trả về tổng các phần tử từ vị trí `left` đến `right` (**đánh số từ 1**, tính cả hai đầu) của mảng mới, lấy **modulo 10⁹ + 7**.

**Ví dụ 1:** `nums = [1,2,3,4], n = 4, left = 1, right = 5` → `13`. Các tổng: `1,3,6,10,2,5,9,3,7,4` → sắp xếp `[1,2,3,3,4,5,6,7,9,10]`; tổng vị trí 1..5 = `1+2+3+3+4 = 13`.

**Ví dụ 2:** `nums = [1,2,3,4], n = 4, left = 3, right = 4` → `6` (= 3 + 3).

**Ví dụ 3:** `nums = [1,2,3,4], n = 4, left = 1, right = 10` → `50`.

**Ràng buộc:** `n == len(nums)`, `1 ≤ n ≤ 1000`, `1 ≤ nums[i] ≤ 100`, `1 ≤ left ≤ right ≤ n(n+1)/2`.

In [1]:
import heapq
import random
import time
from bisect import bisect_right
from typing import List

MOD = 10**9 + 7
examples = [
    (([1, 2, 3, 4], 4, 1, 5), 13),
    (([1, 2, 3, 4], 4, 3, 4), 6),
    (([1, 2, 3, 4], 4, 1, 10), 50),
]
for args, want in examples:
    print(args, "->", want)

([1, 2, 3, 4], 4, 1, 5) -> 13
([1, 2, 3, 4], 4, 3, 4) -> 6
([1, 2, 3, 4], 4, 1, 10) -> 50


## 3. Kiến thức nền

### 3.1 Tổng chạy để sinh tổng đoạn con

Cố định điểm đầu `i`, cộng dần `nums[j]` khi `j` tăng: mỗi bước cho ngay tổng `nums[i..j]` trong `O(1)`.

In [2]:
nums = [1, 2, 3, 4]
for i in range(len(nums)):
    s = 0
    row = []
    for j in range(i, len(nums)):
        s += nums[j]
        row.append(s)
    print(f"bắt đầu tại {i}: {row}")

bắt đầu tại 0: [1, 3, 6, 10]
bắt đầu tại 1: [2, 5, 9]
bắt đầu tại 2: [3, 7]
bắt đầu tại 3: [4]


### 3.2 Chỉ số 1-indexed và slicing

Vị trí `left..right` (từ 1) ứng với slice `arr[left-1:right]` trong Python (slice bỏ đầu mút phải).

In [3]:
arr = [1, 2, 3, 3, 4, 5, 6, 7, 9, 10]
left, right = 3, 4
print(arr[left - 1:right], sum(arr[left - 1:right]))

[3, 3] 6


### 3.3 `heapq` — min-heap

`heapq.heappush(h, x)`, `heapq.heappop(h)` lấy phần tử **nhỏ nhất**. Phần tử là tuple thì so theo thành phần đầu, hoà thì so tiếp thành phần sau.

In [4]:
h = []
for x in [(5, "e"), (1, "a"), (3, "c"), (1, "b")]:
    heapq.heappush(h, x)
print([heapq.heappop(h) for _ in range(len(h))])

[(1, 'a'), (1, 'b'), (3, 'c'), (5, 'e')]


### 3.4 Prefix sum và "prefix của prefix"

`P[k] = nums[0] + … + nums[k−1]` ⇒ tổng `nums[i..j−1] = P[j] − P[i]`.
`Q[k] = P[0] + … + P[k−1]` ⇒ `Σ_{t=i}^{j−1} P[t] = Q[j] − Q[i]`. Dùng `Q` để cộng nhanh **nhiều tổng đoạn** cùng điểm cuối.

In [5]:
from itertools import accumulate
nums = [1, 2, 3, 4]
P = [0] + list(accumulate(nums))
Q = [0] + list(accumulate(P))
print("P =", P)
print("Q =", Q)
j, i = 4, 1
direct = sum(P[j] - P[t] for t in range(i, j))
fast = (j - i) * P[j] - (Q[j] - Q[i])
print("tổng các đoạn kết thúc tại j-1=3, bắt đầu từ i..j-1:", direct, fast)

P = [0, 1, 3, 6, 10]
Q = [0, 0, 1, 4, 10, 20]
tổng các đoạn kết thúc tại j-1=3, bắt đầu từ i..j-1: 20 20


### 3.5 Modulo

Đề yêu cầu `% (10**9 + 7)`. Trong Python số nguyên không tràn, nên chỉ cần lấy mod **ở cuối** (hoặc trong vòng lặp để số nhỏ, kết quả như nhau).

In [6]:
big = 10**12 + 12345
print(big % MOD, (big - 10**12) % MOD + (10**12) % MOD, ((10**12 % MOD) + 12345) % MOD)

5345 1000005352 5345


## 4. Ý tưởng nền tảng / trực giác

Bài này có hai phần tách rời:
1. **Biết được các tổng theo thứ tự tăng** (ít nhất là đến vị trí `right`).
2. **Cộng** đoạn `[left, right]`.

Mẹo quan trọng: gọi `F(k)` = tổng của **k tổng nhỏ nhất**. Khi đó đáp án = `F(right) − F(left − 1)`.

Tính tay với `nums = [1,2,3,4]`, bảng tổng (hàng = điểm đầu, cột = điểm cuối):

| i \ j | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| 0 | 1 | 3 | 6 | 10 |
| 1 | | 2 | 5 | 9 |
| 2 | | | 3 | 7 |
| 3 | | | | 4 |

Mỗi hàng **tăng dần** (vì số dương) → đây là `n` danh sách đã sắp xếp → có thể **trộn** bằng heap. Sắp xếp toàn bộ: `[1,2,3,3,4,5,6,7,9,10]`; `F(5) = 13`, `F(0) = 0` → Ví dụ 1 = 13; `F(4) − F(2) = 9 − 3 = 6` → Ví dụ 2.

In [7]:
nums = [1, 2, 3, 4]
sums = sorted(sum(nums[i:j + 1]) for i in range(4) for j in range(i, 4))
F = [0] + list(accumulate(sums))
print("sorted:", sums)
print("F     :", F)
print("Ví dụ 1:", F[5] - F[0], "| Ví dụ 2:", F[4] - F[2], "| Ví dụ 3:", F[10] - F[0])

sorted: [1, 2, 3, 3, 4, 5, 6, 7, 9, 10]
F     : [0, 1, 3, 6, 9, 13, 18, 24, 31, 40, 50]
Ví dụ 1: 13 | Ví dụ 2: 6 | Ví dụ 3: 50


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** sinh mọi tổng đoạn con bằng tổng chạy (`O(n²)`), sắp xếp, cộng slice `[left−1 : right]`.

(Bản "ngây thơ" hơn nữa là `sum(nums[i:j+1])` cho mỗi đoạn — `O(n³)`, với `n = 1000` là `~1.7·10⁸` phép, quá chậm.)

In [8]:
def brute(nums, n, left, right):
    sums = []
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            sums.append(s)
    sums.sort()
    return sum(sums[left - 1:right]) % MOD

for args, want in examples:
    print(brute(*args), "kỳ vọng", want)

13 kỳ vọng 13
6 kỳ vọng 6
50 kỳ vọng 50


In [9]:
class SolutionSort:
    def rangeSum(self, nums: list[int], n: int, left: int, right: int) -> int:
        sums = []
        for i in range(n):
            s = 0
            for j in range(i, n):
                s += nums[j]
                sums.append(s)
        sums.sort()
        return sum(sums[left - 1:right]) % (10**9 + 7)

print([SolutionSort().rangeSum(*a) for a, _ in examples])

[13, 6, 50]


**Độ phức tạp:** `M = n(n+1)/2 ≈ 5·10⁵` tổng. Time `O(n² log n)` (sắp xếp), Space `O(n²)`.

Đây thực ra là lời giải **được chấp nhận** trên LeetCode (gợi ý của đề cũng vậy). **Nút thắt:** bộ nhớ `O(n²)` và việc sắp xếp mọi tổng, kể cả những tổng ở sau vị trí `right` mà ta không cần. Nếu `n` lớn hơn (ví dụ `10⁵`) thì không còn khả thi.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — các hàng đã sắp xếp sẵn:** vì `nums[i] > 0`, với `i` cố định, `sum(i..j)` tăng nghiêm ngặt theo `j`. Ta có `n` danh sách tăng → **k-way merge** bằng heap: lấy dần `right` phần tử nhỏ nhất.

**Insight 2 — miền giá trị nhỏ:** mọi tổng nằm trong `[1, 100·n] ≤ 10⁵` → đếm số lần xuất hiện của từng giá trị (counting sort), không cần so sánh.

**Insight 3 — tìm theo giá trị:** thay vì liệt kê, hỏi: "có bao nhiêu tổng `≤ x`, và tổng của chúng là bao nhiêu?" — trả lời được trong `O(n)` bằng **hai con trỏ** (vì số dương ⇒ cửa sổ trượt đơn điệu). Binary search `x` nhỏ nhất có `count(x) ≥ k` rồi:

`F(k) = sumLE(x) − (count(x) − k) · x`

**Quy tắc:** *nếu* có nhiều tổng bằng đúng `x` *thì* `count(x)` có thể vượt `k` — phải trừ bớt `(count(x) − k)` bản sao của `x`.

## 7. Các cách tiếp theo

### Cách 2 — Heap k-way merge `O(right · log n)`

1. Đẩy `(nums[i], i, i)` cho mọi `i` — phần tử nhỏ nhất của mỗi "hàng".
2. Lặp `k = 1..right`: pop `(s, i, j)` nhỏ nhất. Nếu `k ≥ left` thì cộng `s`. Nếu `j + 1 < n` đẩy `(s + nums[j+1], i, j+1)`.

In [10]:
def heap_merge(nums, n, left, right):
    h = [(nums[i], i, i) for i in range(n)]
    heapq.heapify(h)
    ans = 0
    for k in range(1, right + 1):
        s, i, j = heapq.heappop(h)
        if k >= left:
            ans += s
        if j + 1 < n:
            heapq.heappush(h, (s + nums[j + 1], i, j + 1))
    return ans % MOD

for args, want in examples:
    print(heap_merge(*args), "kỳ vọng", want)

13 kỳ vọng 13
6 kỳ vọng 6
50 kỳ vọng 50


In [11]:
class SolutionHeap:
    def rangeSum(self, nums: list[int], n: int, left: int, right: int) -> int:
        h = [(nums[i], i) for i in range(n)]       # (tổng, chỉ số cuối)
        heapq.heapify(h)
        ans = 0
        for k in range(1, right + 1):
            s, j = heapq.heappop(h)
            if k >= left:
                ans += s
            if j + 1 < n:
                heapq.heappush(h, (s + nums[j + 1], j + 1))
        return ans % (10**9 + 7)

print([SolutionHeap().rangeSum(*a) for a, _ in examples])

[13, 6, 50]


Ở bản nộp ta bỏ được chỉ số đầu `i` vì chỉ cần biết chỉ số cuối `j` để mở rộng.

**Độ phức tạp:** Time `O(n + right · log n)` (worst `O(n² log n)` khi `right` = `n(n+1)/2`), Space `O(n)` — tốt hơn Cách 1 về bộ nhớ.

### Cách 3 — Counting sort theo giá trị `O(n² + S)`

`S = sum(nums) ≤ 10⁵`. Đếm `cnt[v]` = số đoạn có tổng `v`, rồi quét `v` tăng dần, lấy dần các vị trí thuộc `[left, right]`.

In [12]:
def counting(nums, n, left, right):
    S = sum(nums)
    cnt = [0] * (S + 1)
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            cnt[s] += 1
    ans, pos = 0, 0                     # pos = số phần tử đã đi qua
    for v in range(1, S + 1):
        c = cnt[v]
        if c == 0:
            continue
        lo, hi = pos + 1, pos + c       # các vị trí (1-indexed) mang giá trị v
        a, b = max(lo, left), min(hi, right)
        if a <= b:
            ans += (b - a + 1) * v
        pos += c
        if pos >= right:
            break
    return ans % MOD

for args, want in examples:
    print(counting(*args), "kỳ vọng", want)

13 kỳ vọng 13
6 kỳ vọng 6
50 kỳ vọng 50


Mỗi giá trị `v` chiếm một khối vị trí liên tiếp `[pos+1, pos+c]`; giao khối này với `[left, right]` là phần cần cộng.

**Độ phức tạp:** Time `O(n² + S)`, Space `O(S)`. Không có `log`, nhưng vẫn phải duyệt `n²` đoạn.

### Cách 4 — Binary search trên giá trị + hai con trỏ `O(n log S)` (nâng cao)

**Hàm `count_and_sum(x)`:** với mỗi điểm cuối `j` (prefix `P[j]`), con trỏ `i` là chỉ số nhỏ nhất sao cho `P[j] − P[i] ≤ x`. Vì `P` tăng, `i` chỉ tiến. Các đoạn `[t, j)` với `t ∈ [i, j)` đều có tổng `≤ x`:
- số đoạn: `j − i`
- tổng: `(j − i)·P[j] − (Q[j] − Q[i])`.

In [13]:
def count_and_sum(P, Q, x):
    n = len(P) - 1
    cnt = tot = 0
    i = 0
    for j in range(1, n + 1):
        while P[j] - P[i] > x:
            i += 1
        cnt += j - i
        tot += (j - i) * P[j] - (Q[j] - Q[i])
    return cnt, tot

P = [0] + list(accumulate([1, 2, 3, 4]))
Q = [0] + list(accumulate(P))
for x in [0, 2, 3, 5, 10]:
    print(x, count_and_sum(P, Q, x))

0 (0, 0)
2 (2, 3)
3 (4, 9)
5 (6, 18)
10 (10, 50)


**Hàm `F(k)`:** binary search `x` nhỏ nhất trong `[0, S]` sao cho `count(x) ≥ k`, rồi `F(k) = sumLE(x) − (count(x) − k)·x`. Với `k = 0` thì `F = 0`.

In [14]:
def F_k(P, Q, k):
    if k == 0:
        return 0
    lo, hi = 0, P[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_and_sum(P, Q, mid)[0] >= k:
            hi = mid
        else:
            lo = mid + 1
    c, t = count_and_sum(P, Q, lo)
    return t - (c - k) * lo

def bs_two_pointers(nums, n, left, right):
    P = [0] + list(accumulate(nums))
    Q = [0] + list(accumulate(P))
    return (F_k(P, Q, right) - F_k(P, Q, left - 1)) % MOD

for args, want in examples:
    print(bs_two_pointers(*args), "kỳ vọng", want)

13 kỳ vọng 13
6 kỳ vọng 6
50 kỳ vọng 50


In [15]:
class SolutionBS:
    def rangeSum(self, nums: list[int], n: int, left: int, right: int) -> int:
        P = [0] * (n + 1)
        for i, v in enumerate(nums):
            P[i + 1] = P[i] + v
        Q = [0] * (n + 2)
        for i in range(n + 1):
            Q[i + 1] = Q[i] + P[i]

        def count_sum(x):
            cnt = tot = i = 0
            for j in range(1, n + 1):
                while P[j] - P[i] > x:
                    i += 1
                cnt += j - i
                tot += (j - i) * P[j] - (Q[j] - Q[i])
            return cnt, tot

        def F(k):
            if k == 0:
                return 0
            lo, hi = 0, P[n]
            while lo < hi:
                mid = (lo + hi) // 2
                if count_sum(mid)[0] >= k:
                    hi = mid
                else:
                    lo = mid + 1
            c, t = count_sum(lo)
            return t - (c - k) * lo

        return (F(right) - F(left - 1)) % (10**9 + 7)

print([SolutionBS().rangeSum(*a) for a, _ in examples])

[13, 6, 50]


**Độ phức tạp:** mỗi lần `count_sum` là `O(n)` (hai con trỏ), binary search `O(log S)` lần, gọi 2 lần → Time `O(n log S)`, Space `O(n)`. Đây là cách duy nhất còn chạy được nếu `n` lên tới `10⁵`.

## 8. TRACE từng bước

Trace Cách 2 (heap) với `nums = [1,2,3,4]`, `left = 3`, `right = 4`:

| k | pop (tổng, j) | cộng? | đẩy thêm | heap sau |
|---|---|---|---|---|
| 1 | (1, 0) | không (k < 3) | (3, 1) | (2,1),(3,1),(3,2),(4,3) |
| 2 | (2, 1) | không | (5, 2) | (3,1),(3,2),(4,3),(5,2) |
| 3 | (3, 1) | +3 | (6, 2) | (3,2),(4,3),(5,2),(6,2) |
| 4 | (3, 2) | +3 | (7, 3) | … |

Đáp án `6`.

In [16]:
def trace_heap(nums, left, right):
    n = len(nums)
    h = [(nums[i], i) for i in range(n)]
    heapq.heapify(h)
    ans = 0
    for k in range(1, right + 1):
        s, j = heapq.heappop(h)
        add = k >= left
        if add:
            ans += s
        pushed = None
        if j + 1 < n:
            pushed = (s + nums[j + 1], j + 1)
            heapq.heappush(h, pushed)
        print(f"k={k} pop=({s},{j}) cộng={'có' if add else 'không'} đẩy={pushed} heap={sorted(h)} ans={ans}")
    return ans

print("kết quả:", trace_heap([1, 2, 3, 4], 3, 4))

k=1 pop=(1,0) cộng=không đẩy=(3, 1) heap=[(2, 1), (3, 1), (3, 2), (4, 3)] ans=0
k=2 pop=(2,1) cộng=không đẩy=(5, 2) heap=[(3, 1), (3, 2), (4, 3), (5, 2)] ans=0
k=3 pop=(3,1) cộng=có đẩy=(6, 2) heap=[(3, 2), (4, 3), (5, 2), (6, 2)] ans=3
k=4 pop=(3,2) cộng=có đẩy=(7, 3) heap=[(4, 3), (5, 2), (6, 2), (7, 3)] ans=6
kết quả: 6


Trace Cách 4: binary search tìm `x` cho `k = 5` với `nums = [1,2,3,4]` (sorted sums `[1,2,3,3,4,5,…]`):

In [17]:
def trace_bs(nums, k):
    P = [0] + list(accumulate(nums))
    Q = [0] + list(accumulate(P))
    lo, hi = 0, P[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        c, t = count_and_sum(P, Q, mid)
        print(f"lo={lo} hi={hi} mid={mid} count(mid)={c} sumLE={t} -> {'hi=mid' if c >= k else 'lo=mid+1'}")
        if c >= k:
            hi = mid
        else:
            lo = mid + 1
    c, t = count_and_sum(P, Q, lo)
    res = t - (c - k) * lo
    print(f"x={lo}, count={c}, sumLE={t}, F({k}) = {t} - ({c}-{k})*{lo} = {res}")
    return res

trace_bs([1, 2, 3, 4], 5)
print()
trace_bs([1, 2, 3, 4], 3)       # có hai tổng bằng 3 -> phải trừ bớt

lo=0 hi=10 mid=5 count(mid)=6 sumLE=18 -> hi=mid
lo=0 hi=5 mid=2 count(mid)=2 sumLE=3 -> lo=mid+1
lo=3 hi=5 mid=4 count(mid)=5 sumLE=13 -> hi=mid
lo=3 hi=4 mid=3 count(mid)=4 sumLE=9 -> lo=mid+1
x=4, count=5, sumLE=13, F(5) = 13 - (5-5)*4 = 13

lo=0 hi=10 mid=5 count(mid)=6 sumLE=18 -> hi=mid
lo=0 hi=5 mid=2 count(mid)=2 sumLE=3 -> lo=mid+1
lo=3 hi=5 mid=4 count(mid)=5 sumLE=13 -> hi=mid
lo=3 hi=4 mid=3 count(mid)=4 sumLE=9 -> hi=mid
x=3, count=4, sumLE=9, F(3) = 9 - (4-3)*3 = 6


6

## 9. Code nộp LeetCode

Bản chính: sinh + sắp xếp (ngắn, rõ, chắc chắn AC). Bản tối ưu nhất là `SolutionBS` ở Cách 4 — cũng có thể nộp.

In [18]:
class Solution:
    def rangeSum(self, nums: list[int], n: int, left: int, right: int) -> int:
        sums = []
        for i in range(n):
            s = 0
            for j in range(i, n):
                s += nums[j]
                sums.append(s)
        sums.sort()
        return sum(sums[left - 1:right]) % (10**9 + 7)

for args, want in examples:
    assert Solution().rangeSum(*args) == want
print("OK ví dụ đề")

OK ví dụ đề


## 10. Vì sao thuật toán đúng

**Cách 1:** liệt kê đủ `n(n+1)/2` đoạn (mỗi cặp `i ≤ j` đúng một lần), sắp xếp, cộng đúng slice → hiển nhiên đúng.

**Cách 2 (bất biến heap):** tại mỗi thời điểm, heap chứa đúng **một** "đại diện chưa lấy nhỏ nhất" của mỗi hàng `i` còn phần tử. Vì mỗi hàng tăng dần, phần tử nhỏ nhất chưa lấy toàn cục phải là đại diện của một hàng nào đó → pop cho đúng phần tử thứ `k` trong thứ tự sắp xếp.

**Cách 3:** các tổng bằng nhau đứng liền nhau trong mảng sắp xếp; khối giá trị `v` chiếm đúng `cnt[v]` vị trí sau mọi giá trị nhỏ hơn.

**Cách 4:** `count(x)` không giảm theo `x`. Gọi `x*` nhỏ nhất có `count(x*) ≥ k`. Khi đó phần tử thứ `k` trong mảng sắp xếp bằng `x*` (vì `count(x* − 1) < k`). `k` phần tử nhỏ nhất gồm mọi tổng `< x*` (có `count(x*−1)` cái) và `k − count(x*−1)` bản sao của `x*`. Tổng của chúng = `sumLE(x*) − (count(x*) − k)·x*`. Hai con trỏ đúng vì `P` tăng nghiêm ngặt (số dương) nên với `j` lớn hơn, chỉ số bắt đầu hợp lệ nhỏ nhất không lùi lại. ∎

## 11. Phân tích Time / Space

`n ≤ 1000`, `M = n(n+1)/2 ≤ 500500`, `S = sum(nums) ≤ 10⁵`.

| Cách | Time | Space |
|---|---|---|
| 1. Sinh + sort | `O(n² log n)` | `O(n²)` |
| 2. Heap | `O(n + right·log n)`; worst `O(n² log n)` | `O(n)` |
| 3. Counting | `O(n² + S)` | `O(S)` |
| 4. BS + hai con trỏ | `O(n log S)` | `O(n)` |

Thực tế trong Python, Cách 1 nhanh nhờ `list.sort` viết bằng C; Cách 4 nhanh nhất về tiệm cận.

## 12. Bộ test

Ví dụ đề, edge case (`n = 1`, mọi phần tử bằng nhau — nhiều tổng trùng, `left = right`), random so khớp với brute, và test lớn `n = 1000` giá trị 100 (kiểm tra modulo).

In [19]:
METHODS = {
    "sort": lambda a: SolutionSort().rangeSum(*a),
    "heap": lambda a: SolutionHeap().rangeSum(*a),
    "counting": lambda a: counting(*a),
    "bs": lambda a: SolutionBS().rangeSum(*a),
}
edge = [
    (([5], 1, 1, 1), 5),
    (([2, 2], 2, 2, 3), 6),
    (([1, 1, 1], 3, 1, 6), 10),
    (([7, 1], 2, 3, 3), 8),
    (([100] * 4, 4, 4, 4), 100),
]
for args, want in examples + edge:
    for name, f in METHODS.items():
        got = f(args)
        assert got == want, (name, args, got, want)
print("ví dụ + edge: OK")

ví dụ + edge: OK


In [20]:
rng = random.Random(1508)
for _ in range(500):
    n = rng.randint(1, 15)
    nums = [rng.randint(1, rng.choice([3, 10, 100])) for _ in range(n)]
    M = n * (n + 1) // 2
    left = rng.randint(1, M)
    right = rng.randint(left, M)
    args = (nums, n, left, right)
    ref = brute(*args)
    for name, f in METHODS.items():
        assert f(args) == ref, (name, args)
print("500 test ngẫu nhiên: mọi cách khớp brute")

500 test ngẫu nhiên: mọi cách khớp brute


In [21]:
n = 1000
big = [100] * n
M = n * (n + 1) // 2
args = (big, n, 1, M)
exact = sum(100 * L * (n - L + 1) for L in range(1, n + 1))   # đoạn dài L xuất hiện n-L+1 lần
print("tổng thật:", exact, "> MOD?", exact > MOD)
for name in ["sort", "counting", "bs"]:
    t = time.perf_counter()
    got = METHODS[name](args)
    print(f"{name:8s}: {got} ({time.perf_counter() - t:.2f}s)")
    assert got == exact % MOD
rng = random.Random(9)
nums = [rng.randint(1, 100) for _ in range(1000)]
args = (nums, 1000, 1234, 400000)
t = time.perf_counter()
r1, r2 = METHODS["sort"](args), METHODS["bs"](args)
print("random n=1000:", r1, r2, f"({time.perf_counter() - t:.2f}s)")
assert r1 == r2

tổng thật: 16716700000 > MOD? True


sort    : 716699888 (0.07s)


counting: 716699888 (0.03s)
bs      : 716699888 (0.00s)


random n=1000: 778764762 778764762 (0.10s)


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên chỉ số bắt đầu từ 1

`sum(sums[left:right])` lệch một vị trí so với `sums[left-1:right]`.

In [22]:
def wrong_index(nums, n, left, right):
    sums = sorted(sum(nums[i:j + 1]) for i in range(n) for j in range(i, n))
    return sum(sums[left:right]) % MOD               # SAI: phải là left-1

for args, want in examples:
    print(args, "| sai:", wrong_index(*args), "| đúng:", want)

([1, 2, 3, 4], 4, 1, 5) | sai: 12 | đúng: 13
([1, 2, 3, 4], 4, 3, 4) | sai: 3 | đúng: 6
([1, 2, 3, 4], 4, 1, 10) | sai: 49 | đúng: 50


### Lỗi 2 — Trong binary search, quên trừ các bản sao thừa của `x`

`F(k) = sumLE(x)` là sai khi có nhiều tổng bằng `x`.

In [23]:
def wrong_F(nums, k):
    P = [0] + list(accumulate(nums))
    Q = [0] + list(accumulate(P))
    if k == 0:
        return 0
    lo, hi = 0, P[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_and_sum(P, Q, mid)[0] >= k:
            hi = mid
        else:
            lo = mid + 1
    return count_and_sum(P, Q, lo)[1]                 # SAI: thiếu - (count - k) * x

nums = [1, 2, 3, 4]
print("F(3) sai:", wrong_F(nums, 3), "| đúng:", sum(sorted([1, 3, 6, 10, 2, 5, 9, 3, 7, 4])[:3]))

F(3) sai: 9 | đúng: 6


### Lỗi 3 — Heap: chỉ pop `right − left + 1` lần

Phải pop **từ đầu** (`right` lần) và chỉ cộng khi `k ≥ left`; pop ít hơn là lấy nhầm những tổng nhỏ nhất.

In [24]:
def wrong_heap(nums, n, left, right):
    h = [(nums[i], i) for i in range(n)]
    heapq.heapify(h)
    ans = 0
    for _ in range(right - left + 1):                 # SAI: bỏ qua bước "bỏ đi left-1 phần tử đầu"
        s, j = heapq.heappop(h)
        ans += s
        if j + 1 < n:
            heapq.heappush(h, (s + nums[j + 1], j + 1))
    return ans % MOD

args = ([1, 2, 3, 4], 4, 3, 4)
print("sai:", wrong_heap(*args), "| đúng:", Solution().rangeSum(*args))

sai: 3 | đúng: 6


### Lỗi 4 — Quên modulo

Tổng có thể vượt `10⁹ + 7` (ví dụ `n = 1000`, mọi số 100, lấy hết).

In [25]:
def wrong_mod(nums, n, left, right):
    sums = []
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            sums.append(s)
    sums.sort()
    return sum(sums[left - 1:right])                  # SAI: thiếu % MOD

args = ([100] * 1000, 1000, 1, 500500)
print("sai:", wrong_mod(*args), "| đúng:", Solution().rangeSum(*args))

sai: 16716700000 | đúng: 716699888


### Lỗi 5 — Sinh tổng bằng `sum(nums[i:j+1])` (đúng nhưng `O(n³)`)

Kết quả đúng nhưng chậm gấp hàng trăm lần với `n = 1000`. Ta đo trên `n = 300` để thấy chênh lệch.

In [26]:
nums = [random.Random(3).randint(1, 100) for _ in range(300)]
t = time.perf_counter()
s1 = sorted(sum(nums[i:j + 1]) for i in range(300) for j in range(i, 300))
t1 = time.perf_counter() - t
t = time.perf_counter()
s2 = []
for i in range(300):
    s = 0
    for j in range(i, 300):
        s += nums[j]
        s2.append(s)
s2.sort()
t2 = time.perf_counter() - t
print(f"O(n^3): {t1:.3f}s | tổng chạy O(n^2): {t2:.3f}s | cùng kết quả: {s1 == s2}")

O(n^3): 0.042s | tổng chạy O(n^2): 0.005s | cùng kết quả: True


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Phần tử thứ k / k phần tử nhỏ nhất trong tập ngầm định cực lớn"** → (a) heap k-way merge nếu có các danh sách đã sắp xếp; (b) binary search trên **giá trị** + đếm `≤ x` (LC 378, 668, 719, 1918).
- **Mảng số dương + tổng đoạn** → hai con trỏ / cửa sổ trượt luôn đơn điệu.
- **Tổng trên đoạn `[left, right]` của dãy sắp xếp** → `F(right) − F(left − 1)`.
- **Miền giá trị nhỏ** → counting sort.
- Ràng buộc `n ≤ 1000` ⇒ `O(n² log n)` là đủ: viết Cách 1 trước, tối ưu sau nếu được hỏi.

## 15. Biến thể của bài

1. **LC 1918 – Kth Smallest Subarray Sum** (Premium): chỉ cần `x*` của Cách 4.
2. **LC 719 – Find K-th Smallest Pair Distance**: binary search + hai con trỏ.
3. **LC 378 – Kth Smallest Element in a Sorted Matrix**: heap hoặc binary search trên giá trị.
4. **LC 373 – Find K Pairs with Smallest Sums**: heap k-way merge.
5. Nhiều truy vấn `(left, right)` trên cùng `nums` → tiền xử lý prefix của mảng tổng đã sắp xếp, mỗi truy vấn `O(1)`.
6. `nums` có số âm → hàng không còn tăng dần, heap merge và hai con trỏ hỏng; phải quay về sort.

In [27]:
# Minh hoạ biến thể 1: phần tử thứ k (kth smallest subarray sum) bằng binary search
def kth_smallest(nums, k):
    P = [0] + list(accumulate(nums))
    Q = [0] + list(accumulate(P))
    lo, hi = 0, P[-1]
    while lo < hi:
        mid = (lo + hi) // 2
        if count_and_sum(P, Q, mid)[0] >= k:
            hi = mid
        else:
            lo = mid + 1
    return lo

print([kth_smallest([1, 2, 3, 4], k) for k in range(1, 11)])

[1, 2, 3, 3, 4, 5, 6, 7, 9, 10]


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Sinh + sort | Liệt kê mọi tổng, sắp xếp, cộng slice | `O(n² log n)` | `O(n²)` | Mặc định với `n ≤ 1000` |
| 2. Heap merge | `n` hàng tăng dần, pop `right` lần | `O(right log n)` | `O(n)` | `right` nhỏ, cần ít bộ nhớ |
| 3. Counting | Đếm theo giá trị tổng | `O(n² + S)` | `O(S)` | Giá trị nhỏ, tránh `log` |
| 4. BS + hai con trỏ | `F(k)` qua `count/sum ≤ x` | `O(n log S)` | `O(n)` | `n` lớn, phỏng vấn nâng cao |

## 17. Bài tập vận dụng

### Bài 1 (đoán output)
`Solution().rangeSum([2, 2], 2, 1, 2)` và `Solution().rangeSum([3, 1], 2, 2, 3)` trả về bao nhiêu?

### Bài 2 (tính tay)
Với `nums = [3, 1, 2]`: liệt kê và sắp xếp mọi tổng đoạn con. Tính đáp án với `left = 2, right = 5`.

### Bài 3 (điền TODO)
Viết `count_le(nums, x)`: số đoạn con có tổng `≤ x` bằng **cửa sổ trượt** (không dùng prefix).

### Bài 4 (tự viết lại)
Viết `kth_sum(nums, k)`: tổng đoạn con nhỏ thứ `k` bằng **heap** (không binary search).

### Bài 5 (mini challenge)
Viết `multi_query(nums, queries)`: `queries` là list các `(left, right)`; trả list đáp án (mod 10⁹+7), mỗi truy vấn `O(1)` sau tiền xử lý.

In [28]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def count_le(nums, x):
    # TODO: con trỏ trái lo, tổng cửa sổ win; với mỗi hi: win += nums[hi];
    #       while win > x: win -= nums[lo]; lo += 1; cộng (hi - lo + 1)
    pass

# Bài 4 — TODO
def kth_sum(nums, k):
    # TODO: heap (nums[i], i); pop k lần, lần cuối là đáp án
    pass

# Bài 5 — TODO
def multi_query(nums, queries):
    # TODO: sums đã sort -> prefix F; đáp án (F[r] - F[l-1]) % MOD
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `[2,2]` có tổng `2, 2, 4`. (2) `[3,1]` có tổng `3, 1, 4` → sắp xếp `[1,3,4]`. (3) Đáp án `4` và `7`.

**Bài 2.** (1) 6 đoạn. (2) Tổng: `3, 4, 6, 1, 3, 2`. (3) Sắp xếp `[1,2,3,3,4,6]`, vị trí 2..5 → `2+3+3+4 = 12`.

**Bài 3.** (1) Số dương ⇒ thu hẹp trái khi tổng quá `x`. (2) Mỗi `hi` có `hi − lo + 1` đoạn hợp lệ kết thúc tại đó. (3) Mỗi chỉ số vào/ra cửa sổ tối đa 1 lần ⇒ `O(n)`.

**Bài 4.** (1) Giống Cách 2. (2) Pop `k` lần. (3) Trả giá trị pop lần thứ `k`.

**Bài 5.** (1) Sinh và sort một lần. (2) Prefix `F[0] = 0`, `F[i] = F[i−1] + sums[i−1]`. (3) Mỗi truy vấn là một phép trừ.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
a1 = Solution().rangeSum([2, 2], 2, 1, 2)
a2 = Solution().rangeSum([3, 1], 2, 2, 3)
print("Bài 1:", a1, a2)
assert (a1, a2) == (4, 7)

# Bài 2
sums = sorted(sum([3, 1, 2][i:j + 1]) for i in range(3) for j in range(i, 3))
print("Bài 2:", sums, sum(sums[1:5]))
assert sums == [1, 2, 3, 3, 4, 6] and Solution().rangeSum([3, 1, 2], 3, 2, 5) == 12

Bài 1: 4 7
Bài 2: [1, 2, 3, 3, 4, 6] 12


In [30]:
# Bài 3
def count_le_sol(nums, x):
    lo = win = cnt = 0
    for hi, v in enumerate(nums):
        win += v
        while win > x:
            win -= nums[lo]
            lo += 1
        cnt += hi - lo + 1
    return cnt

# Bài 4
def kth_sum_sol(nums, k):
    n = len(nums)
    h = [(nums[i], i) for i in range(n)]
    heapq.heapify(h)
    for _ in range(k):
        s, j = heapq.heappop(h)
        if j + 1 < n:
            heapq.heappush(h, (s + nums[j + 1], j + 1))
    return s

rng = random.Random(77)
for _ in range(300):
    n = rng.randint(1, 12)
    nums = [rng.randint(1, 20) for _ in range(n)]
    all_s = sorted(sum(nums[i:j + 1]) for i in range(n) for j in range(i, n))
    x = rng.randint(0, sum(nums) + 2)
    assert count_le_sol(nums, x) == sum(1 for s in all_s if s <= x)
    k = rng.randint(1, len(all_s))
    assert kth_sum_sol(nums, k) == all_s[k - 1] == kth_smallest(nums, k)
print("Bài 3, 4: OK")

Bài 3, 4: OK


In [31]:
# Bài 5
def multi_query_sol(nums, queries):
    n = len(nums)
    sums = []
    for i in range(n):
        s = 0
        for j in range(i, n):
            s += nums[j]
            sums.append(s)
    sums.sort()
    F = [0]
    for v in sums:
        F.append(F[-1] + v)
    return [(F[r] - F[l - 1]) % MOD for l, r in queries]

nums = [1, 2, 3, 4]
qs = [(1, 5), (3, 4), (1, 10), (10, 10)]
print("Bài 5:", multi_query_sol(nums, qs))
assert multi_query_sol(nums, qs) == [13, 6, 50, 10]
for _ in range(100):
    n = rng.randint(1, 10)
    nums = [rng.randint(1, 100) for _ in range(n)]
    M = n * (n + 1) // 2
    qs = []
    for _ in range(5):
        l = rng.randint(1, M)
        qs.append((l, rng.randint(l, M)))
    assert multi_query_sol(nums, qs) == [Solution().rangeSum(nums, n, l, r) for l, r in qs]
print("Đáp án Bài 1–5: mọi assert đều qua")

Bài 5: [13, 6, 50, 10]
Đáp án Bài 1–5: mọi assert đều qua


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template sinh tổng đoạn con O(n²):**

```python
for i in range(n):
    s = 0
    for j in range(i, n):
        s += nums[j]          # tổng nums[i..j]
```

**Template "tổng k phần tử nhỏ nhất" bằng binary search giá trị:**

```python
x = min x sao cho count(x) >= k
F(k) = sumLE(x) - (count(x) - k) * x
answer = F(right) - F(left - 1)
```

**Checklist trước khi submit:**
- [ ] Slice `[left - 1 : right]` (chỉ số từ 1).
- [ ] Lấy `% (10**9 + 7)`.
- [ ] Heap: pop đủ `right` lần, chỉ cộng khi `k ≥ left`.
- [ ] Binary search: trừ bản sao thừa của `x` khi tổng trùng.
- [ ] Không dùng `sum(nums[i:j+1])` trong vòng lặp đôi.
- [ ] Đúng chữ ký: `rangeSum(self, nums: list[int], n: int, left: int, right: int) -> int`.